# Pratica 02 — Riproducibilità

**Corso:** MLOps e Monitoraggio con Python · **Lezione 02** · **Tempo stimato:** 40-55 minuti
**Notebook di lezione:** `notebooks_clase/clase02_reproducibilidad_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio parte da un frammento di codice con **lo stesso tipo di problema** che hai corretto nel notebook di lezione, su dati e modelli piccoli giocattolo.
- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Librerie: `numpy`, `pandas`, `PyYAML` e `scikit-learn` (`RandomForestClassifier`, `train_test_split`).

---
## Cella 0 — Setup

In [ ]:
import os
import inspect
import tempfile

import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

print("Librerie caricate correttamente.")

## Esercizio 1 — Una riga di `requirements.txt`

`linea_requirements` deve restituire una riga con il formato esatto di `requirements.txt` (`paquete==version`), a partire da due parametri. Completala.

In [ ]:
def linea_requirements(paquete, version):
    return paquete      # TODO: restituisci f"{paquete}=={version}"

# Controllo automatico
assert linea_requirements("numpy", "2.2.6") == "numpy==2.2.6", "Completa le righe contrassegnate con TODO"
assert linea_requirements("pandas", "3.0.6") == "pandas==3.0.6", "Completa le righe contrassegnate con TODO"
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Generare dati senza toccare lo stato globale

`generar_datos_reproducible` usa `np.random.rand` direttamente (lo stato globale di NumPy), come la cella 4 del notebook sporco. Completala perché usi un generatore proprio, `np.random.default_rng(seed)`, così due chiamate con lo stesso `seed` diano lo stesso risultato.

In [ ]:
def generar_datos_reproducible(n, seed):
    return np.random.rand(n)      # TODO: usa rng = np.random.default_rng(seed) e rng.random(n)

# Controllo automatico
a = generar_datos_reproducible(10, seed=1)
b = generar_datos_reproducible(10, seed=1)
assert np.array_equal(a, b), "Completa le righe contrassegnate con TODO"
c = generar_datos_reproducible(10, seed=2)
assert not np.array_equal(a, c), "con semi diversi anche il risultato dovrebbe essere diverso"
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Caricare la configurazione da un file YAML

`cargar_config` deve leggere un file YAML con `yaml.safe_load` e restituire il dizionario risultante. Per ora non fa nulla. Completala.

In [ ]:
def cargar_config(ruta):
    return {}      # TODO: apri «ruta» e restituisci yaml.safe_load(file)

# Controllo automatico
config_prueba = {"n_filas": 100, "semillas": {"datos": 1, "split": 2, "modelo": 3}}
carpeta_temp = tempfile.mkdtemp()
ruta_temp = os.path.join(carpeta_temp, "config.yaml")
with open(ruta_temp, "w") as f:
    yaml.safe_dump(config_prueba, f)
resultado = cargar_config(ruta_temp)
assert resultado == config_prueba, "Completa le righe contrassegnate con TODO"
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Un modello con tutti i parametri in `config`

`entrenar_modelo_configurable` crea un `RandomForestClassifier` con `n_estimators` e `random_state` scritti direttamente nel codice, ignorando `config`. Completala perché prenda `n_estimators`, `max_depth` e `random_state` (il seme del modello) da `config["hiperparametros"]` e `config["semillas"]["modelo"]`.

In [ ]:
def entrenar_modelo_configurable(X, y, config):
    modelo = RandomForestClassifier(n_estimators=10, random_state=0)      # TODO: usa config["hiperparametros"] e config["semillas"]["modelo"]
    modelo.fit(X, y)
    return modelo

# Controllo automatico
config_prueba = {"hiperparametros": {"n_estimators": 15, "max_depth": 3}, "semillas": {"modelo": 5}}
X_juguete = np.arange(40).reshape(20, 2)
y_juguete = np.array([0, 1] * 10)
m1 = entrenar_modelo_configurable(X_juguete, y_juguete, config_prueba)
assert m1.get_params()["n_estimators"] == 15, "Completa le righe contrassegnate con TODO"
assert m1.get_params()["max_depth"] == 3, "Completa le righe contrassegnate con TODO"
assert m1.get_params()["random_state"] == 5, "Completa le righe contrassegnate con TODO"
m2 = entrenar_modelo_configurable(X_juguete, y_juguete, config_prueba)
assert list(m1.predict(X_juguete)) == list(m2.predict(X_juguete)), "con la stessa config le predizioni devono coincidere"
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Creare la struttura minima di cartelle

`crear_estructura` deve creare, dentro di `base`, le cartelle `src`, `config` e `tests` (con `os.makedirs(..., exist_ok=True)`, perché non fallisca se esistono già) e restituire i loro nomi ordinati alfabeticamente.

In [ ]:
def crear_estructura(base):
    return []      # TODO: crea src/, config/ e tests/ sotto «base», restituisci sorted(["src", "config", "tests"])

# Controllo automatico
base_temp = tempfile.mkdtemp()
resultado = crear_estructura(base_temp)
assert resultado == ["config", "src", "tests"], "Completa le righe contrassegnate con TODO"
for carpeta in ("src", "config", "tests"):
    assert os.path.isdir(os.path.join(base_temp, carpeta)), "manca la cartella " + carpeta
crear_estructura(base_temp)      # non deve fallire se esistono già
print("Esercizio 5 corretto ✔")

---
## Sfida finale — una `pipeline` che integra dati, config e modello

Completa `pipeline_reproducible_practica(config)`, che deve:

- generare i dati con `generar_datos_reproducible(config["n_filas"], config["semillas"]["datos"])` (esercizio 2),
- dividere con `train_test_split(X, y, test_size=config["test_size"], random_state=config["semillas"]["split"])`,
- addestrare con `entrenar_modelo_configurable(X_train, y_train, config)` (esercizio 4),
- restituire `(modelo, exactitud, predicciones)` sul set di test.

`X` è un `DataFrame` con una sola colonna (`pd.DataFrame({"x": datos})`) e `y` sono le etichette: `(datos > 0.5).astype(int)`.

In [ ]:
def pipeline_reproducible_practica(config):
    datos = None      # TODO: generar_datos_reproducible(config["n_filas"], config["semillas"]["datos"])
    X = pd.DataFrame({"x": datos})
    y = (datos > 0.5).astype(int)
    X_train, X_test, y_train, y_test = None, None, None, None      # TODO: train_test_split con test_size e random_state da config
    modelo = None      # TODO: entrenar_modelo_configurable(X_train, y_train, config)
    exactitud = None      # TODO: modelo.score(X_test, y_test)
    predicciones = None      # TODO: modelo.predict(X_test)
    return modelo, exactitud, predicciones

# Controllo automatico
config_reto = {
    "n_filas": 60, "test_size": 0.25,
    "hiperparametros": {"n_estimators": 10, "max_depth": 3},
    "semillas": {"datos": 11, "split": 11, "modelo": 11},
}
resultado = pipeline_reproducible_practica(config_reto)
assert resultado is not None, "Completa le righe contrassegnate con TODO"
modelo, exactitud, predicciones = resultado
assert hasattr(modelo, "predict"), "restituisci il modello addestrato, non None"
assert isinstance(exactitud, float), "restituisci l'accuratezza come numero"
modelo_b, exactitud_b, predicciones_b = pipeline_reproducible_practica(config_reto)
assert exactitud == exactitud_b, "con la stessa config l'accuratezza deve ripetersi"
assert list(predicciones) == list(predicciones_b), "con la stessa config le predizioni devono ripetersi"
print("Sfida finale corretta ✔")

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
def linea_requirements(paquete, version):
    return f"{paquete}=={version}"

# Controllo automatico
assert linea_requirements("numpy", "2.2.6") == "numpy==2.2.6", "Completa le righe contrassegnate con TODO"
assert linea_requirements("pandas", "3.0.6") == "pandas==3.0.6", "Completa le righe contrassegnate con TODO"
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
def generar_datos_reproducible(n, seed):
    rng = np.random.default_rng(seed)
    return rng.random(n)

# Controllo automatico
a = generar_datos_reproducible(10, seed=1)
b = generar_datos_reproducible(10, seed=1)
assert np.array_equal(a, b), "Completa le righe contrassegnate con TODO"
c = generar_datos_reproducible(10, seed=2)
assert not np.array_equal(a, c), "con semi diversi anche il risultato dovrebbe essere diverso"
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
def cargar_config(ruta):
    with open(ruta) as archivo:
        return yaml.safe_load(archivo)

# Controllo automatico
config_prueba = {"n_filas": 100, "semillas": {"datos": 1, "split": 2, "modelo": 3}}
carpeta_temp = tempfile.mkdtemp()
ruta_temp = os.path.join(carpeta_temp, "config.yaml")
with open(ruta_temp, "w") as f:
    yaml.safe_dump(config_prueba, f)
resultado = cargar_config(ruta_temp)
assert resultado == config_prueba, "Completa le righe contrassegnate con TODO"
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
def entrenar_modelo_configurable(X, y, config):
    modelo = RandomForestClassifier(
        n_estimators=config["hiperparametros"]["n_estimators"],
        max_depth=config["hiperparametros"]["max_depth"],
        random_state=config["semillas"]["modelo"],
    )
    modelo.fit(X, y)
    return modelo

# Controllo automatico
config_prueba = {"hiperparametros": {"n_estimators": 15, "max_depth": 3}, "semillas": {"modelo": 5}}
X_juguete = np.arange(40).reshape(20, 2)
y_juguete = np.array([0, 1] * 10)
m1 = entrenar_modelo_configurable(X_juguete, y_juguete, config_prueba)
assert m1.get_params()["n_estimators"] == 15, "Completa le righe contrassegnate con TODO"
assert m1.get_params()["max_depth"] == 3, "Completa le righe contrassegnate con TODO"
assert m1.get_params()["random_state"] == 5, "Completa le righe contrassegnate con TODO"
m2 = entrenar_modelo_configurable(X_juguete, y_juguete, config_prueba)
assert list(m1.predict(X_juguete)) == list(m2.predict(X_juguete)), "con la stessa config le predizioni devono coincidere"
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
def crear_estructura(base):
    carpetas = ["src", "config", "tests"]
    for carpeta in carpetas:
        os.makedirs(os.path.join(base, carpeta), exist_ok=True)
    return sorted(carpetas)

# Controllo automatico
base_temp = tempfile.mkdtemp()
resultado = crear_estructura(base_temp)
assert resultado == ["config", "src", "tests"], "Completa le righe contrassegnate con TODO"
for carpeta in ("src", "config", "tests"):
    assert os.path.isdir(os.path.join(base_temp, carpeta)), "manca la cartella " + carpeta
crear_estructura(base_temp)      # non deve fallire se esistono già
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
def pipeline_reproducible_practica(config):
    datos = generar_datos_reproducible(config["n_filas"], config["semillas"]["datos"])
    X = pd.DataFrame({"x": datos})
    y = (datos > 0.5).astype(int)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=config["test_size"], random_state=config["semillas"]["split"])
    modelo = entrenar_modelo_configurable(X_train, y_train, config)
    exactitud = modelo.score(X_test, y_test)
    predicciones = modelo.predict(X_test)
    return modelo, exactitud, predicciones

# Controllo automatico
config_reto = {
    "n_filas": 60, "test_size": 0.25,
    "hiperparametros": {"n_estimators": 10, "max_depth": 3},
    "semillas": {"datos": 11, "split": 11, "modelo": 11},
}
resultado = pipeline_reproducible_practica(config_reto)
assert resultado is not None, "Completa le righe contrassegnate con TODO"
modelo, exactitud, predicciones = resultado
assert hasattr(modelo, "predict"), "restituisci il modello addestrato, non None"
assert isinstance(exactitud, float), "restituisci l'accuratezza come numero"
modelo_b, exactitud_b, predicciones_b = pipeline_reproducible_practica(config_reto)
assert exactitud == exactitud_b, "con la stessa config l'accuratezza deve ripetersi"
assert list(predicciones) == list(predicciones_b), "con la stessa config le predizioni devono ripetersi"
print("Sfida finale corretta ✔")

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto della lezione non ti è rimasto chiaro (dipendenze bloccate, semi, struttura di cartelle, separare la configurazione)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*